In [2]:
# ================================
# PHASE 1: DATA ARCHITECTURE
# ================================

import numpy as np
import pandas as pd
import time

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Generate dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    weights=[0.9, 0.1],   # 90% placed, 10% unplaced
    random_state=42
)

# 2. Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Feature scaling (avoid data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ================================
# PHASE 2: BASELINE MODEL
# ================================

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred = rf.predict(X_test_scaled)

baseline_acc = accuracy_score(y_test, y_pred)
baseline_f1 = f1_score(y_test, y_pred)

print("\n--- BASELINE MODEL ---")
print("Accuracy:", baseline_acc)
print("F1 Score:", baseline_f1)


# ================================
# GRID SEARCH (ACCURACY)
# ================================

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_acc = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid_acc.fit(X_train_scaled, y_train)

print("\n--- GRID SEARCH (ACCURACY) ---")
print("Best Params:", grid_acc.best_params_)
print("Best Score:", grid_acc.best_score_)


# ================================
# GRID SEARCH (F1 SCORE)
# ================================

grid_f1 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

start = time.time()
grid_f1.fit(X_train_scaled, y_train)
grid_time = time.time() - start

print("\n--- GRID SEARCH (F1) ---")
print("Best Params:", grid_f1.best_params_)
print("Best F1 Score:", grid_f1.best_score_)
print("Time Taken:", grid_time, "seconds")


# ================================
# RANDOMIZED SEARCH
# ================================

param_dist = {
    'n_estimators': np.arange(10, 500),
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': np.arange(2, 20)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)

start = time.time()
random_search.fit(X_train_scaled, y_train)
random_time = time.time() - start

print("\n--- RANDOMIZED SEARCH ---")
print("Best Params:", random_search.best_params_)
print("Best F1 Score:", random_search.best_score_)
print("Time Taken:", random_time, "seconds")


# ================================
# FINAL COMPARISON TABLE
# ================================

results = pd.DataFrame({
    "Method": ["Grid Search", "Randomized Search"],
    "Time Taken (sec)": [grid_time, random_time],
    "Best F1 Score": [grid_f1.best_score_, random_search.best_score_]
})

print("\n--- FINAL COMPARISON ---")
print(results)


--- BASELINE MODEL ---
Accuracy: 0.915
F1 Score: 0.2608695652173913

--- GRID SEARCH (ACCURACY) ---
Best Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Best Score: 0.915

--- GRID SEARCH (F1) ---
Best Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Best F1 Score: 0.300098731677679
Time Taken: 21.672657012939453 seconds

--- RANDOMIZED SEARCH ---
Best Params: {'n_estimators': np.int64(225), 'min_samples_split': np.int64(5), 'max_depth': None}
Best F1 Score: 0.2642424242424242
Time Taken: 41.3608775138855 seconds

--- FINAL COMPARISON ---
              Method  Time Taken (sec)  Best F1 Score
0        Grid Search         21.672657       0.300099
1  Randomized Search         41.360878       0.264242
